# LangChain

This notebook provides hands-on practice to become familiar with **LangChain** and the core concepts used to build LLM-powered applications.

## What is LangChain?

**LangChain** is a framework for building applications powered by large language models (LLMs).

It provides structured components for working with prompts, models, tools, memory, and external data sources, making it easier to create real-world LLM use cases such as chatbots, agents, and retrieval-augmented generation (RAG) systems.

## Installations & Imports

In [ ]:
# uncomment and run if for some reason there is no packages in env
# !pip -q install langchain langchain_openai langchain-community huggingface_hub openai

In [1]:
!pip show langchain

Name: langchain
Version: 1.2.13
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /opt/miniconda3/envs/mlpeople5/lib/python3.11/site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


In [ ]:
%load_ext autoreload
%autoreload 2

# LLMs
from langchain_openai import ChatOpenAI

# Prompts
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder

# Messages
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage

# Output parsers (important for LCEL)
from langchain_core.output_parsers import StrOutputParser

# Runnables (LCEL)
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

import os

overall_temperature = 0.1

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### API keys

In scope of this notebook next API keys are used and must be set in env variables:
 - `OPENAI_API_KEY` - https://platform.openai.com/settings/organization/api-keys
 - `SERPAPI_API_KEY` - https://serpapi.com/dashboard
 - `HF_TOKEN` - https://huggingface.co/settings/tokens

## Create Model

Use OpenAI model as example here. Links:
 - https://docs.langchain.com/oss/python/integrations/chat/openai - Langchain ChatOpenAI integration
 - https://reference.langchain.com/python/langchain-openai/chat_models/base/ChatOpenAI - Langchain ChatOpenAI API reference
 - https://platform.openai.com/ - OpenAI Platform docs


In [3]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=overall_temperature,
)

In [7]:
# dispaly default params
llm

ChatOpenAI(profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x10e5db710>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x10f118890>, root_client=<openai.OpenAI object at 0x10e34de10>, root_async_client=<openai.AsyncOpenAI object at 0x10f1183d0>, model_name='gpt-4o-mini', temperature=0.1, model_kwargs={}, openai_api_key=SecretStr('******

### Get Response

llm.invoke returns AIMessage type

In [9]:
llm.invoke("Explain what LangChain is in one sentence")

AIMessage(content='LangChain is a framework designed to facilitate the development of applications that utilize large language models (LLMs) by providing tools for chaining together various components like prompts, memory, and APIs.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 15, 'total_tokens': 52, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DQBVH1VTgOERN5M77D6GfnEvucngW', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d4e2d-d03c-7893-8604-c74e6356babb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 37, 'total_tokens': 52, 'input_token_details': {'au

`StrOutputParser` converts LLM `AIMessage` output into plain text. It is useful in LCEL chains so the text can be cleanly passed to the next step.

In [13]:
outputParser = StrOutputParser() 
outputParser.invoke(llm.invoke("Explain what LangChain is in one sentence"))

'LangChain is a framework designed to facilitate the development of applications that leverage large language models (LLMs) by providing tools for chaining together various components like prompts, memory, and APIs.'

In [16]:
pasta_question_input = "What are 5 vacation destinations for someone who likes to eat pasta?"
pasta_question_output = llm.invoke(pasta_question_input)
outputParser.invoke(pasta_question_output)

"If you love pasta, there are several fantastic vacation destinations where you can indulge in delicious pasta dishes. Here are five great options:\n\n1. **Italy**: The birthplace of pasta, Italy is a must-visit for any pasta lover. Regions like Emilia-Romagna are famous for their handmade pasta, such as tagliatelle and tortellini. Cities like Bologna, Florence, and Naples offer a wide variety of pasta dishes, from classic spaghetti carbonara to regional specialties.\n\n2. **San Francisco, California**: Known for its vibrant food scene, San Francisco boasts numerous Italian restaurants that serve authentic pasta dishes. The North Beach neighborhood is particularly famous for its Italian cuisine, where you can enjoy everything from fresh fettuccine to rich lasagna.\n\n3. **New York City, New York**: NYC is home to a diverse array of Italian restaurants, ranging from classic trattorias to modern eateries. You can find delicious pasta dishes in neighborhoods like Little Italy and the West

In [17]:
print(pasta_question_output.content)

If you love pasta, there are several fantastic vacation destinations where you can indulge in delicious pasta dishes. Here are five great options:

1. **Italy**: The birthplace of pasta, Italy is a must-visit for any pasta lover. Regions like Emilia-Romagna are famous for their handmade pasta, such as tagliatelle and tortellini. Cities like Bologna, Florence, and Naples offer a wide variety of pasta dishes, from classic spaghetti carbonara to regional specialties.

2. **San Francisco, California**: Known for its vibrant food scene, San Francisco boasts numerous Italian restaurants that serve authentic pasta dishes. The North Beach neighborhood is particularly famous for its Italian cuisine, where you can enjoy everything from fresh fettuccine to rich lasagna.

3. **New York City, New York**: NYC is home to a diverse array of Italian restaurants, ranging from classic trattorias to modern eateries. You can find delicious pasta dishes in neighborhoods like Little Italy and the West Villag

In [18]:
def query_llm(question: str) -> AIMessage:
    """
    Sends a question to the ChatOpenAI model, prints the response text, and returns the full AIMessage.

    Args:
        question (str): The text question or prompt to send to the model.

    Returns:
        AIMessage: The model's response object containing content, role, and other metadata.
    """
    # Invoke the model
    response: AIMessage = llm.invoke(question)

    # Print the actual text
    print(response.content)

    return response

In [19]:
query_llm("who am I talking to?");

You’re talking to an AI language model created by OpenAI. I'm here to help answer your questions and provide information on a wide range of topics. How can I assist you today?


### Stream

| `invoke()`              | `stream()`                       |
| ----------------------- | -------------------------------- |
| Waits for full output   | Yields output progressively      |
| Returns one `AIMessage` | Returns many partial chunks      |
| Simple, synchronous     | Real-time, generator-based       |
| Good for chains         | Good for UI / agents / live logs |


In [ ]:
question = "Give me a 5-line poem about pasta."

# Stream the response token by token
for event in llm.stream(question):
    # Each event is a partial update from the model
    if hasattr(event, "content") and event.content:
        print(event.content, end="", flush=True)

print()  # newline at the end

In boiling water, dreams entwine,  
Golden strands in a dance divine.  
With sauce that whispers, herbs that sing,  
Each twirl a joy, each bite a fling.  
Pasta, a love that time can't confine.


In [20]:
for event in llm.stream(
    "What are some theories about the relationship between unemployment and inflation? Give me 3 and make it short"
):
    if hasattr(event, "content") and event.content:
        print(event.content, end="", flush=True)

print()  # newline at the end

Here are three key theories about the relationship between unemployment and inflation:

1. **Phillips Curve**: This theory posits an inverse relationship between unemployment and inflation, suggesting that lower unemployment leads to higher inflation and vice versa. The rationale is that when more people are employed, demand for goods and services increases, driving prices up.

2. **Natural Rate Hypothesis**: This theory argues that there is a "natural" rate of unemployment that the economy tends to return to in the long run. Inflation can be influenced by deviations from this natural rate, but over time, inflation expectations adjust, leading to a neutral relationship between unemployment and inflation.

3. **Supply Shock Theory**: This theory suggests that external shocks (like oil price spikes) can simultaneously increase inflation and unemployment, a phenomenon known as stagflation. In this scenario, rising costs lead to higher prices while economic activity slows, resulting in hig

## Prompts: Managing requests for LLMs

A prompt is the structured input you send to the model to guide how it should respond.

In LangChain, prompts are templates that:
 - inject variables into text,
 - define system / user / assistant roles,
 - make inputs reusable and consistent.

In [21]:
# Basic example PromptTemplate
prompt = PromptTemplate(
    input_variables=["food"],            # variables you will fill later
    template="What are 3 vacation destinations for someone who likes to eat {food}?",
)

print(prompt.format(food="donuts"))

What are 3 vacation destinations for someone who likes to eat donuts?


In [22]:
question_prompt = prompt.format(food="donuts")
response = llm.invoke(question_prompt)
print(response.content)

If you love donuts, here are three vacation destinations that are known for their delicious and unique donut offerings:

1. **Portland, Oregon**: Portland is a donut lover's paradise, famous for its artisanal donut shops. Voodoo Doughnut is a must-visit, known for its quirky flavors and creative toppings. Donut Vault and Blue Star Donuts also offer gourmet options that are sure to satisfy your sweet tooth. The city's vibrant food scene and coffee culture make it a perfect destination for foodies.

2. **Los Angeles, California**: LA boasts a diverse array of donut shops, from classic to innovative. Donut Friend is popular for its customizable donuts and unique flavor combinations. Randy's Donuts, with its iconic rooftop sign, serves up classic favorites and is a staple in the city. The variety of options, including vegan and gluten-free donuts, makes LA a great spot for donut enthusiasts.

3. **New York City, New York**: NYC is home to some of the most famous donut shops in the country.

### Chat-style prompt

In [ ]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that provides short and consize answers"),
    ("human", "Explain {topic} in simple terms.")
])

In [24]:
chat_question_prompt = chat_prompt.format(topic="donuts")
response = llm.invoke(chat_question_prompt)
print(response.content)

Donuts are sweet, fried treats that are usually round with a hole in the middle. They can be made from dough, which is a mixture of flour, sugar, eggs, and milk. After frying, they are often coated with sugar, icing, or filled with things like jelly or cream. Donuts come in many flavors and styles, making them a popular snack or dessert!


## Chains

A chain is a pipeline where the output of one step becomes the input of the next.

In LangChain (LCEL), chains are built with the | operator to connect components like:

prompt → model → output parser → function → …

Why chains
 - Compose multiple steps into one flow
 - Keep logic modular and readable
 - Reuse prompts, models, and parsers
 - Build more complex behavior without agents

In [ ]:
chain = prompt | llm | StrOutputParser()

result = chain.invoke({"food": "pasta"})
print(result)

If you love pasta, here are three fantastic vacation destinations where you can indulge in delicious pasta dishes:

1. **Italy**: The birthplace of pasta, Italy offers a wide variety of regional pasta dishes. You can explore cities like Bologna, known for its rich ragù sauce and tagliatelle; Naples, famous for its spaghetti alle vongole (spaghetti with clams); and Rome, where you can enjoy classic dishes like cacio e pepe and carbonara. Each region has its own specialties, making Italy a pasta lover's paradise.

2. **San Francisco, California, USA**: San Francisco has a vibrant Italian-American community and is home to some excellent Italian restaurants. You can find a range of pasta dishes, from traditional to modern interpretations. The North Beach neighborhood is particularly known for its Italian cuisine, where you can enjoy fresh pasta made with local ingredients.

3. **Buenos Aires, Argentina**: Argentina has a strong Italian influence, and you'll find a variety of pasta dishes t

### RunnablePassthrough

In general, RunnablePassthrough is an old LCEL transition style and outdated.

But it still can be actually useful mot for formatting prompts, but for cases like:
 - branching data
 - injecting extra fields
 - combining multiple inputs
 - pass to output result intermediate chain values

In [33]:
joke_prompt = PromptTemplate(
    input_variables=["topic"],
    template="What is most popular joke related to {topic}?",
)

In [ ]:
# example - Final step contains all 3 - original topic, generated prompt and generated joke
joke_chain = RunnableParallel(
    topic=RunnablePassthrough(lambda topic: topic),
    prompt=RunnablePassthrough(lambda topic: joke_prompt.format(topic=topic)),
    joke=joke_prompt | llm | StrOutputParser(),

)

result = joke_chain.invoke("pasta")
print(result)
print(result["joke"])

{'topic': 'pasta', 'prompt': 'pasta', 'joke': "One of the most popular pasta jokes is:\n\n**Why didn’t the ravioli get invited to hang out with the cool pasta?**\n\n**Because it was too square!** \n\nIt's a classic play on words that pasta lovers enjoy!"}
One of the most popular pasta jokes is:

**Why didn’t the ravioli get invited to hang out with the cool pasta?**

**Because it was too square!** 

It's a classic play on words that pasta lovers enjoy!


## Agents

**Agents** are LLM-powered controllers that can **decide which tools to use**, **in what order**, and **with what inputs** to answer a user’s question.

Unlike simple chains (prompt → model → output), an agent can:

* plan steps,
* call external tools (calculator, Python, web search, APIs),
* observe tool results,
* iterate if needed,
* produce a final answer.

In LangChain v1, agents are built on top of **LangGraph**, where:

* the **model** decides what to do next,
* **tools** execute actions,
* the agent loops until the task is complete.

This enables **reasoning + acting** (ReAct pattern): the model reasons about the problem, acts via tools, observes results, and continues reasoning.


### Agent - Basic Example

In [37]:
from langchain.agents import create_agent
from langchain_core.tools import tool

# create model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=overall_temperature)

# create tools
@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression."""
    return str(eval(expression))

@tool
def python_interpreter(code: str) -> str:
    """Execute Python code. Use print() to output results."""
    local_vars = {}
    exec(code, {}, local_vars)
    return str(local_vars)

# create agent
agent = create_agent(
    model=llm,
    tools=[calculator, python_interpreter],
    system_prompt="You are a helpful assistant that can use tools for math and Python code"
)

### Agent - Explore invoke result

In [ ]:
invoke_result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is 25 * 17?"}]}
)

# display raw agent invoke result
invoke_result

{'messages': [HumanMessage(content='What is 25 * 17?', additional_kwargs={}, response_metadata={}, id='1c57a3af-b02e-4b61-9561-a133713f1a82'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 90, 'total_tokens': 107, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DPql4CyCLWPgGvzA1kILUUUC95lEs', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d496c-fff1-7952-bf04-dfe9491c86c0-0', tool_calls=[{'name': 'calculator', 'args': {'expression': '25 * 17'}, 'id': 'call_BUa4OXyW2gl8SauQIWiBkrP0', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 90, 'output_toke

In [6]:
for key in invoke_result.keys():
    print(key)

messages


In [ ]:
# display messages from agent invoke result
invoke_result["messages"]

[HumanMessage(content='What is 25 * 17?', additional_kwargs={}, response_metadata={}, id='1c57a3af-b02e-4b61-9561-a133713f1a82'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 90, 'total_tokens': 107, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DPql4CyCLWPgGvzA1kILUUUC95lEs', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d496c-fff1-7952-bf04-dfe9491c86c0-0', tool_calls=[{'name': 'calculator', 'args': {'expression': '25 * 17'}, 'id': 'call_BUa4OXyW2gl8SauQIWiBkrP0', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 90, 'output_tokens': 17, 'tota

### Agent - Explore stream result

In [7]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is 25 * 17?"}]},
    stream_mode="updates"
):
    print(chunk, end="\n")

{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 90, 'total_tokens': 107, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_cd4a20171f', 'id': 'chatcmpl-DPqnJ7LzTolmGniHg7b36hAOaCvWI', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d496f-1d9e-7ef2-8f25-81f33ce78707-0', tool_calls=[{'name': 'calculator', 'args': {'expression': '25 * 17'}, 'id': 'call_7wJWXPNSq1mdIh9xmKg5oxzM', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 90, 'output_tokens': 17, 'total_tokens': 107, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 

In [ ]:
# attempt to print result in more human readable way
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is 25 * 17?"}]},
    stream_mode="updates",
    version="v2",
):
    if chunk["type"] == "updates":
        for step, data in chunk["data"].items():
            print(f"step: {step}")
            print(f"content: {data['messages'][-1].content_blocks}")

step: model
content: [{'type': 'tool_call', 'name': 'calculator', 'args': {'expression': '25 * 17'}, 'id': 'call_8LGfZfEolvPrNBnHTk17vh0K'}]
step: tools
content: [{'type': 'text', 'text': '425'}]
step: model
content: [{'type': 'text', 'text': 'The result of \\( 25 \\times 17 \\) is 425.'}]


### Define tools

In [ ]:
# uncomment and run if for some reason there is no packages in env
# !pip install -q google-search-results

In [ ]:
from serpapi import GoogleSearch

@tool
def calculator(expression: str) -> str:
    """Evaluate a simple math expression."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"
    

@tool
def python_interpreter(code: str) -> str:
    """Execute Python code using print() to return results."""
    local_vars = {}
    try:
        exec(code, {}, local_vars)
        return str(local_vars)
    except Exception as e:
        return f"Error: {e}"


@tool
def web_search(query: str) -> str:
    """Search the web using SerpAPI."""
    params = {
        "q": query,
        "api_key": os.getenv("SERPAPI_API_KEY"),
        "engine": "google",
        "num": 3,
    }
    search = GoogleSearch(params)
    results = search.get_dict()
    snippets = [r.get("snippet", "") for r in results.get("organic_results", [])[:3]]
    return "\n".join(snippets)

### Create LLM and agent

In [41]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=overall_temperature, streaming=True)

tools = [calculator, python_interpreter, web_search]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant that can use tools for math, Python code, and web search."
)

In [14]:
user_question_1 = "Calculate sqrt(144) + 20 using the calculator tool. Also search the latest SpaceX news."

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": user_question_1}]},
    stream_mode="updates",
    version="v2",
):
    if chunk["type"] == "updates":
        for step, data in chunk["data"].items():
            print(f"step: {step}")
            print(f"content: {data['messages'][-1].content_blocks}")

step: model
content: [{'type': 'tool_call', 'name': 'calculator', 'args': {'expression': 'sqrt(144) + 20'}, 'id': 'call_ByRUDBDTfiHJosXcKjffKUom'}, {'type': 'tool_call', 'name': 'web_search', 'args': {'query': 'latest SpaceX news'}, 'id': 'call_oTrFzhdGMhHeom817ORQ8nIX'}]
step: tools
content: [{'type': 'text', 'text': "Error: name 'sqrt' is not defined"}]
step: tools
content: [{'type': 'text', 'text': 'SpaceX has acquired xAI to form the most ambitious, vertically-integrated innovation engine on (and off) Earth, with AI, rockets, space-based internet, ...'}]
step: model
content: [{'type': 'tool_call', 'name': 'calculator', 'args': {'expression': '(144)**0.5 + 20'}, 'id': 'call_AfAfNZzK4hVVfQl6v2B0LROq'}, {'type': 'tool_call', 'name': 'web_search', 'args': {'query': 'latest SpaceX news'}, 'id': 'call_m0ZF6Gkt6QwKtlrOZxpSgcPo'}]
step: tools
content: [{'type': 'text', 'text': '32.0'}]
step: tools
content: [{'type': 'text', 'text': 'SpaceX has acquired xAI to form the most ambitious, verti

In [ ]:
agent_invoke_result1 = agent.invoke({"messages": [{"role": "user", "content": user_question_1}]})

In [ ]:
# display final model answer
print(agent_invoke_result1["messages"][-1].content)

The result of the calculation \( \sqrt{144} + 20 \) is \( 32.0 \).

In the latest SpaceX news, the company has acquired xAI to form a highly ambitious, vertically-integrated innovation engine focused on both Earth and space, incorporating AI, rockets, and space-based internet.


### “verbose” demo

parse agent invoke result and display in human readable format

In [44]:
def pretty_print_agent_result(result):
    print("\n=== AGENT TRACE ===\n")

    for msg in result["messages"]:

        if isinstance(msg, HumanMessage):
            print(f"[USER]\n{msg.content}\n")

        elif isinstance(msg, AIMessage):
            # Tool call request (ReAct thinking step)
            if msg.tool_calls:
                for call in msg.tool_calls:
                    print(f"[TOOL CALL] {call['name']}({call['args']})\n")

            # Final LLM response
            elif msg.content.strip():
                print(f"[LLM]\n{msg.content}\n")

        elif isinstance(msg, ToolMessage):
            print(f"[TOOL RESULT] ({msg.name})\n{msg.content}\n")

    print("=== END TRACE ===\n")

In [21]:
pretty_print_agent_result(agent_invoke_result1)


=== AGENT TRACE ===

[USER]
Calculate sqrt(144) + 20 using the calculator tool. Also search the latest SpaceX news.

[TOOL CALL] calculator({'expression': 'sqrt(144) + 20'})

[TOOL CALL] web_search({'query': 'latest SpaceX news'})

[TOOL RESULT] (calculator)
Error: name 'sqrt' is not defined

[TOOL RESULT] (web_search)
SpaceX has acquired xAI to form the most ambitious, vertically-integrated innovation engine on (and off) Earth, with AI, rockets, space-based internet, ...

[TOOL CALL] calculator({'expression': '(144)**0.5 + 20'})

[TOOL CALL] web_search({'query': 'latest SpaceX news'})

[TOOL RESULT] (calculator)
32.0

[TOOL RESULT] (web_search)
SpaceX has acquired xAI to form the most ambitious, vertically-integrated innovation engine on (and off) Earth, with AI, rockets, space-based internet, ...

[LLM]
The result of the calculation \( \sqrt{144} + 20 \) is \( 32.0 \).

In the latest SpaceX news, the company has acquired xAI to form a highly ambitious, vertically-integrated innovati

### Streaming “verbose” demo

parse agent stream result and display in human readable format

In [47]:
def pretty_print_agent_stream(agent, user_question):
    print("\n=== STREAMING AGENT TRACE ===\n")

    for chunk in agent.stream(
        {"messages": [{"role": "user", "content": user_question}]}
    ):
        # Each chunk is: {'model': {...}} OR {'tools': {...}}
        for node_name, node_data in chunk.items():
            messages = node_data.get("messages", [])

            for msg in messages:
                # USER (rarely appears in stream)
                if isinstance(msg, HumanMessage):
                    print(f"[USER]\n{msg.content}\n")

                # LLM output or tool calls
                elif isinstance(msg, AIMessage):
                    if msg.tool_calls:
                        for call in msg.tool_calls:
                            print(f"[TOOL CALL] {call['name']}({call['args']})\n")
                    elif msg.content.strip():
                        print(f"[LLM]\n{msg.content}\n")

                # Tool result
                elif isinstance(msg, ToolMessage):
                    print(f"[TOOL RESULT] ({msg.name})\n{msg.content}\n")

    print("=== END STREAM ===\n")

#### Query - find person & calculate prime

In [46]:
user_question_2 = "Who is the current leader of China? What is the largest prime number that is smaller than their age?"

pretty_print_agent_stream(agent, user_question_2)


=== STREAMING AGENT TRACE ===

[TOOL CALL] web_search({'query': 'current leader of China 2023'})

[TOOL CALL] calculator({'expression': '70 - 1'})

[TOOL RESULT] (calculator)
69

[TOOL RESULT] (web_search)
Xi Jinping (born 15 June 1953) is a Chinese statesman and politician who has served as the general secretary of the Chinese Communist Party (CCP) and ...
Xi Jinping is a politician and government official who became president of China in 2013 and general secretary of the Chinese Communist Party in ...
1983–present ; Xi Jinping 习近平 (born 1953) · 17 March 2018, 10 March 2023 ; Xi Jinping 习近平 (born 1953) · 10 March 2023, Incumbent ...

[TOOL CALL] python_interpreter({'code': 'def is_prime(n):\n    if n <= 1:\n        return False\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0:\n            return False\n    return True\n\nlargest_prime = 0\nfor num in range(68, 1, -1):\n    if is_prime(num):\n        largest_prime = num\n        break\nlargest_prime'})

[TOOL RESULT] (

#### Query - find country population & do some math

In [34]:
user_question_3 = "Find the population of Canada and calculate 3% of it. Then find the area of the circle with the radius value equal to found number"

pretty_print_agent_stream(agent, user_question_3)


=== STREAMING AGENT TRACE ===

[TOOL CALL] web_search({'query': 'current population of Canada 2023'})

[TOOL CALL] calculator({'expression': '3% of x'})

[TOOL RESULT] (calculator)
Error: invalid syntax (<string>, line 1)

[TOOL RESULT] (web_search)
Population of Canada (2026 and historical) ; 2023, 39,299,105, 1.23% ; 2022, 38,821,259, 0.85% ; 2020, 38,171,902, 1.2% ; 2015, 35,962,234, 1.01% ...
Canada's population was estimated at 40,528,396 on October 1, 2023, an increase of 430,635 people (+1.1%) from July 1. This was the highest ...
Total population for Canada in 2023 was 40,097,761, a 2.98% increase from 2022. Total population for Canada in 2022 was 38,939,056, a 1.83% increase from 2021.

[TOOL CALL] calculator({'expression': '3% of 40528396'})

[TOOL RESULT] (calculator)
Error: invalid syntax (<string>, line 1)

[TOOL CALL] calculator({'expression': '0.03 * 40528396'})

[TOOL CALL] calculator({'expression': 'pi * (40528396 ** 2)'})

[TOOL RESULT] (calculator)
1215851.88

[TOOL

#### Query - today's weather & Celsius vs. Fahrenheit

In [35]:
user_question_4 = "Find today’s weather in Tokyo and compute the difference between the temperature in Celsius and Fahrenheit."

pretty_print_agent_stream(agent, user_question_4)


=== STREAMING AGENT TRACE ===

[TOOL CALL] web_search({'query': 'Tokyo weather today'})

[TOOL CALL] calculator({'expression': '(F - 32) * 5/9 = C'})

[TOOL RESULT] (calculator)
Error: invalid syntax (<string>, line 1)

[TOOL RESULT] (web_search)
Hourly Weather · 1 AM 51°. rain drop 73% · 2 AM 51°. rain drop 73% · 3 AM 50°. rain drop 49% · 4 AM 50°. rain drop 49% · 5 AM 50°. rain drop 49% · 6 AM 50°.
Cloudy skies. Low 59F. Winds NW at 5 to 10 mph. Humidity. 82%. UV Index. 0 of 11. Moonrise. 4:17 pm. Moonset. 4: ...

[TOOL CALL] web_search({'query': 'Tokyo weather today temperature in Celsius and Fahrenheit'})

[TOOL RESULT] (web_search)
... becoming windy in the afternoon with times of clouds and sun Hi: 58° · Current Weather. 7:41 PM. 53°F. Rain. RealFeel® 46° · MINUTECAST®. Rain for at least 60 ...
... (GMT +9) | Updated 55 seconds ago. --° | 50°. 52 °F. like 52°. icon. Rain. N. 3. Gusts 4 °mph. Tomorrow's temperature is forecast to be COOLER than today.
Weather in Tokyo, Japan ... 

## Memory

The simplest form of memory is simply passing chat history messages along a thread. Here's an example:

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini")

prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessage(
            content="You are a helpful assistant. Answer all questions to the best of your ability."
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | llm

ai_msg = chain.invoke(
    {
        "messages": [
            HumanMessage(
                content="Translate from English to French: I love programming."
            ),
            AIMessage(content="J'adore la programmation."),
            HumanMessage(content="What did you just say?"),
        ],
    }
)
print(ai_msg)

content='I translated "I love programming" into French, which is "J\'adore la programmation."' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 56, 'total_tokens': 74, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DPrT7gqT6e0jeP0B05tmppAswEfbO', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d4996-ae0a-7842-8e31-07bc79967e2a-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 56, 'output_tokens': 18, 'total_tokens': 74, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [37]:
ai_msg.content

'I translated "I love programming" into French, which is "J\'adore la programmation."'

### Automatic history management

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph
from langchain_core.messages import SystemMessage

# Create a graph workflow whose STATE is a list of chat messages.
# MessagesState is a predefined schema: {"messages": List[BaseMessage]}
workflow = StateGraph(state_schema=MessagesState)


# This function is a GRAPH NODE.
# It receives the current conversation STATE and must return
# an updated piece of state (here: new messages from the model).
def call_model(state: MessagesState):
    # System prompt is re-added on every model call.
    # It is NOT stored in memory — only user/AI messages are.
    system_prompt = (
        "You are a helpful assistant. "
        "Answer all questions to the best of your ability."
    )

    # The model must receive the full conversation history.
    # LangGraph keeps that history inside state["messages"].
    messages = [SystemMessage(content=system_prompt)] + state["messages"]

    # Call the LLM with the conversation so far
    response = llm.invoke(messages)

    # IMPORTANT:
    # A graph node must return a DICT that updates the state.
    # Here we append the model response into the "messages" state.
    return {"messages": response}


# Register the node in the graph under the name "model"
workflow.add_node("model", call_model)

# Define the graph flow:
# START → model
# (Whenever the app is invoked, it starts at START and goes to this node)
workflow.add_edge(START, "model")


# MemorySaver is a CHECKPOINTER.
# It automatically saves and restores the graph state between calls.
# This is the LangGraph way of implementing conversational memory.
memory = MemorySaver()

# Compile the workflow into a runnable app with memory enabled.
# From now on, app.invoke(...) will remember previous messages
# as long as the same thread_id is used.
app = workflow.compile(checkpointer=memory)

In [42]:
config = {"configurable": {"thread_id": "user_1"}}

app.invoke(
    {"messages": [HumanMessage(content="Hi, my name is Maksym. I was born before 2000")]},
    config=config,
)

app.invoke(
    {"messages": [HumanMessage(content="What is my name?")]},
    config=config,
)

app.invoke(
    {"messages": [HumanMessage(content="And what is my age?")]},
    config=config,
)

{'messages': [HumanMessage(content='Hi, my name is Maksym. I was born before 2000', additional_kwargs={}, response_metadata={}, id='9484c8ee-43a1-4ff2-8cde-9501c52dcda7'),
  AIMessage(content="Hi Maksym! It's nice to meet you. Since you were born before 2000, you likely have some interesting experiences and perspectives shaped by growing up in the late 20th century and early 21st century. If you have any questions or topics you'd like to discuss, feel free to ask!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 42, 'total_tokens': 104, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DPrdAdLeFz2GV0EWF5jwJaxnJSPos', 'service_tier': 'default', 'f


**Two types of memory**
| Type          | Where                 | How it works                         | Scope                   |
| ------------- | --------------------- | ------------------------------------ | ----------------------- |
| Prompt memory | `MessagesPlaceholder` | You manually pass messages           | One call only           |
| Graph memory  | `MemorySaver`         | LangGraph stores state between calls | Persistent conversation |


### Modifying Chat History
Modifying saved chat messages can help chatbot handle different situations.

#### Trimming Messages
LLMs and chat models have limited context windows, and even if you don’t exceed the limits, you can limit the number of distractions the model has to deal with. One solution is to trim the history messages before passing them to the model. Let’s look at the example of the history with the app we announced above:

In [44]:
demo_ephemeral_chat_history = [
    HumanMessage(content="Hey there! I'm Nemo."),
    AIMessage(content="Hello!"),
    HumanMessage(content="How are you today?"),
    AIMessage(content="Fine thanks!"),
]

app.invoke(
    {
        "messages": demo_ephemeral_chat_history
        + [HumanMessage(content="What's my name?")]
    },
    config={"configurable": {"thread_id": "user_2"}},
)

{'messages': [HumanMessage(content="Hey there! I'm Nemo.", additional_kwargs={}, response_metadata={}, id='553d30cd-da9f-4456-b688-706160110293'),
  AIMessage(content='Hello!', additional_kwargs={}, response_metadata={}, id='8fa48e1b-637a-4382-8885-95c35c0bbdf6', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='How are you today?', additional_kwargs={}, response_metadata={}, id='1ec48054-0932-4998-9ba5-be6d0bd95a69'),
  AIMessage(content='Fine thanks!', additional_kwargs={}, response_metadata={}, id='67a711e7-36a0-49f6-b6a6-af018d03948d', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content="What's my name?", additional_kwargs={}, response_metadata={}, id='1748f848-5d64-4380-924e-5b4f6afc8a73'),
  AIMessage(content='Your name is Nemo!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 63, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_

We can see that the program remembers the previously loaded name.

But let's imagine that we have a very small context window and we want to reduce the number of messages passed to the model to the last 2. We can use the built-in trim_messages utility to trim the messages based on their number of tokens before they reach our request. In this case, we will count each message as 1 "token" and keep only the last two messages:

In [45]:
from langchain_core.messages import trim_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph
from langchain_core.messages import SystemMessage

# This component trims the conversation history before sending it to the model.
# - strategy="last": keeps the last messages
# - max_tokens=2: only keeps the last 2 messages
# - token_counter=len: counts each message as 1 token (simplified)
# This is useful to limit the context for large conversations.
trimmer = trim_messages(strategy="last", max_tokens=2, token_counter=len)

# Create a LangGraph workflow with the MessagesState schema
workflow = StateGraph(state_schema=MessagesState)

# Node function: called with the current conversation state
def call_model(state: MessagesState):
    # Apply trimming to reduce message history length
    trimmed_messages = trimmer.invoke(state["messages"])

    # System prompt added every call
    system_prompt = (
        "You are a helpful assistant. "
        "Answer all questions to the best of your ability."
    )

    # Concatenate system prompt with trimmed conversation messages
    messages = [SystemMessage(content=system_prompt)] + trimmed_messages

    # Call the LLM with the current context
    response = llm.invoke(messages)

    # Return updated state for the workflow
    return {"messages": response}

# Register node and edge in the graph
workflow.add_node("model", call_model)
workflow.add_edge(START, "model")

# Enable memory: saves conversation state between calls
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

In [47]:
app.invoke(
    {
        "messages": demo_ephemeral_chat_history
        + [HumanMessage(content="What is my name?")]
    },
    config={"configurable": {"thread_id": "user_3"}},
)

{'messages': [HumanMessage(content="Hey there! I'm Nemo.", additional_kwargs={}, response_metadata={}, id='553d30cd-da9f-4456-b688-706160110293'),
  AIMessage(content='Hello!', additional_kwargs={}, response_metadata={}, id='8fa48e1b-637a-4382-8885-95c35c0bbdf6', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='How are you today?', additional_kwargs={}, response_metadata={}, id='1ec48054-0932-4998-9ba5-be6d0bd95a69'),
  AIMessage(content='Fine thanks!', additional_kwargs={}, response_metadata={}, id='67a711e7-36a0-49f6-b6a6-af018d03948d', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='b1ed5687-8faa-4001-9e91-d06ee6f2affa'),
  AIMessage(content="I'm sorry, but I don't have access to personal information about you unless you share it with me. How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 39, 'tot

#### Message History Summaries

We can use this prompt in other cases as well. For example, we can use an additional LLM call to generate a conversation summary before calling our application. Let's play back our chat history:

In [56]:
from langchain_core.messages import HumanMessage, RemoveMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph

# Create the workflow graph with message-based state
workflow = StateGraph(state_schema=MessagesState)

# Node function: called with the current conversation state
def call_model(state: MessagesState):
    # System prompt for the assistant
    system_prompt = (
        "You are a helpful assistant. "
        "Answer all questions to the best of your ability. "
        "The provided chat history includes a summary of the earlier conversation."
    )
    system_message = SystemMessage(content=system_prompt)

    # Get the previous messages except the most recent one
    message_history = state["messages"][:-1]

    # If chat history gets long, summarize older messages
    if len(message_history) >= 4:
        last_human_message = state["messages"][-1]

        # Prompt to summarize past messages
        summary_prompt = (
            "Distill the above chat messages into a single summary message. "
            "Include as many specific details as you can."
        )

        # Call LLM to generate summary of older messages
        summary_message = llm.invoke(
            message_history + [HumanMessage(content=summary_prompt)]
        )

        # Remove older messages from memory (optional cleanup)
        delete_messages = [RemoveMessage(id=m.id) for m in state["messages"]]

        # Keep the latest human message
        human_message = HumanMessage(content=last_human_message.content)

        # Call LLM with system message, summary, and latest human message
        response = llm.invoke([system_message, summary_message, human_message])

        # Update messages: summary + latest human message + new response + cleanup
        message_updates = [summary_message, human_message, response] + delete_messages
    else:
        # If chat history is short, just process all messages normally
        message_updates = llm.invoke([system_message] + state["messages"])

    return {"messages": message_updates}

# Add node and edge to the graph
workflow.add_node("model", call_model)
workflow.add_edge(START, "model")

# Enable memory
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

In [57]:
app.invoke(
    {
        "messages": demo_ephemeral_chat_history
        + [HumanMessage("What did I say my name was?")]
    },
    config={"configurable": {"thread_id": "user_4"}},
)

{'messages': [AIMessage(content='Nemo greeted the assistant and asked how it was doing, to which the assistant responded that it was fine.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 60, 'total_tokens': 82, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DPrupkTFbG0NDtTEMaXbEaAqWmqdE', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d49b0-e3f8-7612-b9c7-8da2dbddbba6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 60, 'output_tokens': 22, 'total_tokens': 82, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
  HumanMess

### Memory Patterns in LangGraph / LangChain

| Pattern | How it Works | Pros | Cons / Notes |
|---------|--------------|------|--------------|
| **Plain MemorySaver** | Stores all messages and passes them to the LLM every time. | Simple to implement, keeps full context. | Can grow very large for long conversations, high token usage. |
| **Memory + Trimming (`trim_messages`)** | Keeps only the last N messages or tokens before calling the LLM. | Reduces token usage, keeps recent context. | Older context is lost, may affect reasoning if needed context is trimmed. |
| **Memory + Summarization (`RemoveMessage` + summary)** | When chat history exceeds a threshold, summarizes older messages into a single summary, deletes old messages, and calls LLM with summary + latest message. | Keeps conversation compact, preserves essential context, scales for long sessions. | Slight overhead for summarization calls; summary may lose fine details. |

#### Key Notes

- **Trimming** is useful for short-lived chats or token-limited LLMs.  
- **Summarization** is better for long-term memory in multi-turn conversations.  
- You can combine both strategies: summarize old messages and trim the very latest ones if token usage is still high.  
- `MemorySaver` acts as a simple checkpointer to store/retrieve message history for your workflow.

## Comparing LLMs with Hugging Face

To compare different language models, we will use models hosted on Hugging Face.

**Hugging Face** provides a large collection of open-source LLMs and the popular **Transformers** library for running them. It also offers a model hub where developers can explore, test, and use pre-trained models for tasks like text generation, classification, and translation.

Browse available text-generation models here:
https://huggingface.co/models?pipeline_tag=text-generation

In [ ]:
# uncomment and run if for some reason there is no packages in env
# ! pip install -q langchain-huggingface

In [7]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

import warnings
warnings.filterwarnings('ignore')

In [9]:
llm_qwen = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    max_new_tokens=512,
    top_k=10,
    top_p=0.95,
    temperature=0.01,
    repetition_penalty=1.03,
    huggingfacehub_api_token=os.environ["HF_TOKEN"],
)

chat_qwen = ChatHuggingFace(llm=llm_qwen)
print(chat_qwen.invoke("What is Deep Learning?").content)

Deep Learning is a subset of machine learning that focuses on artificial neural networks with multiple layers, which can learn and extract complex features from raw data. The term "deep" refers to the depth of the network, meaning it has many layers between the input and output layers. This architecture allows deep learning models to automatically and adaptively learn hierarchical representations of data, making them highly effective for a wide range of tasks such as image and speech recognition, natural language processing, and more.

Key aspects of Deep Learning include:

1. **Artificial Neural Networks (ANNs)**: These are computational models inspired by the structure and function of biological neural networks. ANNs consist of layers of interconnected nodes (neurons) that process information.

2. **Deep Architectures**: These involve multiple layers of abstraction, allowing the model to learn from raw data and automatically discover the features needed for classification or predicti

### Model's tasks

**Main text-related tasks:**

| Task | What it does | Example model |
|---|---|---|
| `text-generation` | Generates text (continues a prompt) | GPT-2, LLaMA base |
| `conversational` | Chat format (system/user/assistant) | Qwen-Instruct, LLaMA-Instruct |
| `text2text-generation` | Input → output (translation, summarization) | T5, BART |
| `summarization` | Compresses text | facebook/bart-large-cnn |
| `translation` | Translates text | Helsinki-NLP/opus-mt |
| `fill-mask` | Fills in missing words | BERT |
| `text-classification` | Classification (sentiment, etc.) | distilbert-base-uncased |
| `token-classification` | NER, POS-tagging | dslim/bert-base-NER |
| `question-answering` | Answers questions from context | deepset/roberta-base-squad2 |
| `feature-extraction` | Embeddings (vectors) | sentence-transformers |

In [11]:
from huggingface_hub import HfApi

def get_task(model_id):
    info = HfApi().model_info(model_id, expand=["inferenceProviderMapping"])
    for p in info.inference_provider_mapping:
        if p.status == "live":
            print(f"  {p.provider}: task={p.task}")

get_task("Qwen/Qwen2.5-7B-Instruct")

  together: task=conversational
  featherless-ai: task=conversational


### Chains with Hugging Face models

It is possible to use HuggingFace models in Chains from Langchain.

In [14]:
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

llm = HuggingFaceEndpoint(
  repo_id="Qwen/Qwen2.5-72B-Instruct",
  huggingfacehub_api_token = os.getenv("HUGGINGFACEHUB_API_TOKEN"),
)

chat_model = ChatHuggingFace(llm=llm)

prompt = ChatPromptTemplate.from_messages([
  SystemMessagePromptTemplate.from_template("You're a helpful assistant."),
  HumanMessagePromptTemplate.from_template("{user_question}"),
])

chain = prompt | chat_model | StrOutputParser()

response = chain.invoke({"user_question": "What happens when an unstoppable force meets an immovable object?"})

print(response)

The question of what happens when an unstoppable force meets an immovable object is a classic thought experiment that explores the limits of logic and physics. In a purely theoretical sense, it presents a paradox because both concepts—unstoppable force and immovable object—are idealizations that cannot coexist in the same universe under the laws of physics as we understand them.

1. **Philosophical Perspective**: From a philosophical standpoint, this scenario is often used to illustrate the nature of paradoxes and the limitations of language and logic. It challenges our understanding of absolute concepts and can lead to discussions about the nature of reality and the limits of human reasoning.

2. **Physical Perspective**: In the realm of physics, the concept of an "unstoppable force" implies a force that can overcome any resistance, while an "immovable object" implies something that cannot be moved by any force. According to the laws of physics, such absolutes do not exist. Every forc

### Setting up a "laboratory" of models to compare their responses

In [ ]:
overal_temperature = 0.1

minimax_llm = HuggingFaceEndpoint(
    repo_id="MiniMaxAI/MiniMax-M2.5",
    temperature=overal_temperature,
    max_new_tokens=500
)
minimax = ChatHuggingFace(llm=minimax_llm)

llama_llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    temperature=overal_temperature,
    max_new_tokens=500
)
llama = ChatHuggingFace(llm=llama_llm)

qwen_llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    temperature=overal_temperature,
    max_new_tokens=500
)
qwen = ChatHuggingFace(llm=qwen_llm)

In [16]:
# alternative of old ModelLaboratory
models = {
    "MiniMax-M2.5": minimax,
    "Llama-3.1-8B": llama,
    "Qwen-2.5-7B": qwen,
}

In [17]:
def compare(prompt, models=models, **kwargs):
    formatted = prompt.format(**kwargs)
    for name, model in models.items():
        print(f"\n{'='*50}")
        print(f"🔹 {name}")
        print('='*50)
        try:
            response = model.invoke(formatted)
            print(response.content)
        except Exception as e:
            print(f"❌ Error: {e}")

In [18]:
template1 = """Question: {question}

Answer: Let's think step by step."""
prompt1 = PromptTemplate(template=template1, input_variables=["question"])


In [19]:
compare(prompt1, question="What is the opposite of up?")


🔹 MiniMax-M2.5
The opposite of "up" is **down**.

This is a simple directional antonym pair:

- Up = toward a higher position
- Down = toward a lower position

🔹 Llama-3.1-8B
To find the opposite of "up," let's consider the direction in the three-dimensional space. 

1. In a vertical direction, "up" is the opposite of "down."
2. In a horizontal direction, "up" is the opposite of "left" or "right," depending on the context.
3. In a circular direction, "up" is the opposite of "down" or "bottom."

However, if we're looking for a single word that is the opposite of "up" in a general sense, the most common answer would be "down."

🔹 Qwen-2.5-7B
The opposite of "up" is "down." Let's break this down step by step:

1. **Understanding "Up":** "Up" generally refers to a direction that is opposite to the pull of gravity, or the direction towards the sky or the top of something. It can also be used metaphorically to mean improvement, elevation, or increase.

2. **Identifying the Opposite:** The o

In [20]:
compare(prompt1, question= "The cafeteria had 23 apples. \
If they used 20 for lunch, and bought 6 more, how many apple do they have?")


🔹 MiniMax-M2.5
**Step‑by‑step solution**

1. Start with the original number of apples: **23**.
2. Subtract the apples used for lunch:  
   \(23 - 20 = 3\) apples remain.
3. Add the apples they bought:  
   \(3 + 6 = 9\) apples.

**Answer:** The cafeteria now has **9 apples**.

🔹 Llama-3.1-8B
To find out how many apples the cafeteria has, let's break it down step by step:

1. Initially, the cafeteria had 23 apples.
2. They used 20 apples for lunch, so we subtract 20 from 23:
   23 - 20 = 3
   Now, they have 3 apples left.
3. Then, they bought 6 more apples. To find the total number of apples, we add 6 to 3:
   3 + 6 = 9
   Now, the cafeteria has 9 apples.

So, the cafeteria has 9 apples.

🔹 Qwen-2.5-7B
Sure, let's break it down step by step:

1. **Initial number of apples**: The cafeteria started with 23 apples.
2. **Apples used for lunch**: They used 20 apples for lunch. So, we subtract 20 from the initial 23 apples:
   \[
   23 - 20 = 3
   \]
   After using 20 apples, they have 3 app

In [21]:
compare(prompt1,
        question="Can Elon Musk have a conversation with George Washington?")


🔹 MiniMax-M2.5
No, Elon Musk cannot have a conversation with George Washington. Here's why:

1. **Time difference**: George Washington died in 1799, while Elon Musk was born in 1971. They lived over 200 years apart.

2. **Physical impossibility**: For a direct conversation to occur, both parties need to be alive at the same time. This is not the case here.

3. **No existing technology**: While there are historical recordings, writings, and AI recreations of historical figures, no technology currently exists that would allow a living person to have a real-time conversation with someone who died centuries ago.

The only ways to "interact" with George Washington would be:
- Reading his writings and letters
- Visiting historical sites or museums about him
- Using AI chatbots designed to simulate historical figures (which are simulations, not actual conversations)

So in reality, an actual conversation between the two is impossible.

🔹 Llama-3.1-8B
To determine if Elon Musk can have a conv

In [22]:
template2 = """You are a professional social media manager who
can write great posts in linkedin to increase appeal of persons profile: {request}

Story:"""
prompt2 = PromptTemplate(template=template2, input_variables=["request"])

In [23]:
compare(prompt2, request="I have passed a course in large language models (1 month duration). Write a post about that.")


🔹 MiniMax-M2.5
Here's a professional LinkedIn post for your course completion:

---

🚀 Just completed an intensive one-month journey into Large Language Models! 🤖

Over the past month, I've dive deep into the fascinating world of LLMs—from understanding transformer architectures to exploring prompt engineering, fine-tuning techniques, and real-world applications.

Key takeaways:
✅ How LLMs process and generate human-like text
✅ Prompt engineering strategies for optimal results
✅ Practical applications across industries
✅ Ethical considerations and responsible AI use

This field is evolving rapidly, and I'm excited to apply these insights to [your field/industry]. The potential of AI to transform how we work, create, and solve problems is truly remarkable.

Grateful to the instructors and peers who made this learning journey so valuable. Looking forward to connecting with others passionate about AI and exploring how LLMs can drive innovation!

#AI #MachineLearning #LLM #LargeLanguageMo

In [24]:
template3 = """Answer the question to the best of your abilities but
if you are not sure then answer you don't know: {question}

Answer:"""
prompt3 = PromptTemplate(template=template3, input_variables=["question"])

In [25]:
compare(
    prompt3,
    question="I am riding a bicycle. The pedals are moving fast. I look into the mirror and I am not moving. Why is this?"
)



🔹 MiniMax-M2.5
We need to answer the question: "I am riding a bicycle. The pedals are moving fast. I look into the mirror and I am not moving. Why is this?" The user wants an answer. The user says "Answer the question to the best of your abilities but if you are not sure then answer you don't know: I am riding a bicycle. The pedals are moving fast. I look into the mirror and I am not moving. Why is this?" So they want an answer. The scenario: The person is riding a bicycle, the pedals are moving fast, they look into the mirror and they appear not moving. Why? The mirror is presumably attached to the bike (like a rear-view mirror) or maybe they are looking at a side mirror? The scenario: The person is moving relative to the ground, but relative to the bike they are stationary. The mirror is attached to the bike, so relative to the bike they are stationary. So they see themselves not moving in the mirror because they are stationary relative to the mirror. The mirror moves with the bike,

### Named Entity Recognition (NER)
This is a separate interesting task for which we can also use the same conversational model. We can also choose a model trained for this specific task.

In [26]:
template4 = """{question}

Answer:"""
prompt4 = PromptTemplate(template=template4, input_variables=["question"])

In [27]:
compare(
    prompt4,
    question="""
    Extract names and cities from the text.\n\n
    Output in the format: {"names": list of names in text, "cities": list of cities in text}\n\n

    Mark Dickey, 40, began suffering from severe gastric pain last week after descending
    more than 3,600 feet into Morca Cave in Southern Turkey, according to a statement
    from the European Cave Rescue Association, but he wasn’t able to be rescued until yesterday,
    when doctors deemed him “transportable.
    """
)


🔹 MiniMax-M2.5
{"names": ["Mark Dickey"], "cities": []}

🔹 Llama-3.1-8B
{"names": ["Mark Dickey"], "cities": ["Southern Turkey"]}

🔹 Qwen-2.5-7B
```json
{"names": ["Mark Dickey"], "cities": []}
```

Explanation: The text mentions "Mark Dickey" as a name, but it does not mention any specific cities.


### Answers to text-based questions
Let's see how different models handle this task.

In [28]:
compare(prompt4,
    question="""Is Mark Dickey alive?\n\n
Output in the format: Yes or No, facts that prove that.\n\n

Mark Dickey, 40, began suffering from severe gastric pain last week after descending
more than 3,600 feet into Morca Cave in Southern Turkey, according to a statement
from the European Cave Rescue Association, but he wasn’t able to be rescued until yesterday,
when doctors deemed him “transportable.""")


🔹 MiniMax-M2.5
The user asks: "Is Mark Dickey alive?" They provide a short snippet: Mark Dickey, 40, began suffering from severe gastric pain last week after descending more than 3,600 feet into Morca Cave in Southern Turkey, according to a statement from the European Cave Rescue Association, but he wasn’t able to be rescued until yesterday, when doctors deemed him “transportable."

We need to answer: Yes or No, facts that prove that.

We need to determine if Mark Dickey is alive. The snippet says he was rescued and was transportable. It doesn't explicitly say he survived. However, we need to check if there is any known news about his condition. The snippet is likely from a news article about his rescue. The question: "Is Mark Dickey alive?" The answer likely is Yes, he was rescued and is alive. But we need to confirm with facts. The snippet says he was rescued and doctors deemed him transportable. That implies he was alive at that point. However, we need to be careful: The snippet mi